In [1]:
import gc
import os

import numpy as np
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback
from benchmarks.cnn import CNN
from dataset.pubchemqc import PubChemQCDataset

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "pubchemqc_homo",
    "pubchemqc_lumo",
    "pubchemqc_hl_gap",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
]
num_tasks = len(target_properties)

model_path_template = "../models/cnn_1.1M-pac_grid-[PubChemQC]-fold_{fold}"
model_logs_dir = "../logs"
model_logs_template = "cnn_1.1M-pac_grid-[PubChemQC]-fold_{fold}"

dataset = PubChemQCDataset(
    properties=target_properties, require_quality_flags=False, num_workers=20
)

targets_array = np.array(dataset.targets, dtype=np.float32)

categories = np.array(
    [f"{pac.category}_{pac.count_total_rings}" for pac in dataset.pacs]
)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_results = []

W1006 23:01:13.451000 16552 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda


In [2]:
for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), categories), 1
):
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=categories[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    scaler = MultiTargetScaler()
    train_targets_np = targets_array[train_idx]
    scaler.fit(train_targets_np)

    all_targets_scaled = scaler.transform(targets_array)
    dataset.targets = torch.tensor(all_targets_scaled, dtype=torch.float32)

    train_loader = DataLoader(train_subset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=256, shuffle=False)

    model = CNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=92
    )
    lightning_model = LightningModule(model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = LightningModule.load_from_checkpoint(
        best_model_path, model=model, property_names=target_properties, scaler=scaler
    )
    torch.save(
        best_lightning_model.model.state_dict(),
        os.path.join(save_path_dir, "best.ckpt"),
    )

    dataset.targets = targets_array

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_subset, val_subset, train_loader, val_loader
    del model, lightning_model, best_lightning_model
    del trainer, checkpoint_callback, logger, scaler
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")

D:\Coding\pac-gcnn\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.



======[ Fold 1/5 ]======


D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ CNN        │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss    │      0 │ train │     0 │
│ 2 │ train_rmsd │ MetricDict │      0 │ train │     0 │
│ 3 │ val_rmsd   │ MetricDict │      0 │ train │     0 │
│ 4 │ test_rmsd  │ MetricDict │      0 │ train │     0 │
│ 5 │ test_mae   │ MetricDict │      0 │ train │     0 │
│ 6 │ test_r2    │ MetricDict │      0 │ train │     0 │
│ 7 │ test_max   │ MetricDict │      0 │ train │     0 │
│ 8 │ test_min   │ MetricDict │      0 │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.494                                                                      
Modules in train mode: 47                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.494                                                                      
Modules in train mode: 47                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


Epoch 001 | Scaled Train Loss: N/A | Scaled Val Loss: 1.95071 | pubchemqc_homo Val RMSD: 1.62802 | pubchemqc_lumo Val RMSD: 0.84242 | pubchemqc_hl_gap Val RMSD: 0.81964
Epoch 002 | Scaled Train Loss: 0.54328 | Scaled Val Loss: 2.28435 | pubchemqc_homo Val RMSD: 1.62963 | pubchemqc_lumo Val RMSD: 0.80301 | pubchemqc_hl_gap Val RMSD: 1.19691
Epoch 003 | Scaled Train Loss: 0.23154 | Scaled Val Loss: 0.65074 | pubchemqc_homo Val RMSD: 0.62800 | pubchemqc_lumo Val RMSD: 0.55962 | pubchemqc_hl_gap Val RMSD: 0.77597
Epoch 004 | Scaled Train Loss: 0.12620 | Scaled Val Loss: 0.09781 | pubchemqc_homo Val RMSD: 0.25760 | pubchemqc_lumo Val RMSD: 0.20561 | pubchemqc_hl_gap Val RMSD: 0.30165
Epoch 005 | Scaled Train Loss: 0.07737 | Scaled Val Loss: 0.08454 | pubchemqc_homo Val RMSD: 0.20387 | pubchemqc_lumo Val RMSD: 0.21281 | pubchemqc_hl_gap Val RMSD: 0.28463
Epoch 006 | Scaled Train Loss: 0.05294 | Scaled Val Loss: 0.07485 | pubchemqc_homo Val RMSD: 0.18992 | pubchemqc_lumo Val RMSD: 0.19239 | p

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100 | Scaled Train Loss: 0.00473 | Scaled Val Loss: 0.04162 | pubchemqc_homo Val RMSD: 0.12781 | pubchemqc_lumo Val RMSD: 0.13640 | pubchemqc_hl_gap Val RMSD: 0.22075
Best scaled validation loss for fold 5: 0.0409

======[ Final k-fold results ]======
Scaled Loss per fold: [0.0385, 0.0329, 0.0431, 0.0435, 0.0409]
Average Scaled Loss: 0.0398 ± 0.0039


In [3]:
import gc
import os

import numpy as np
import torch
from lightning import pytorch as pl
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset

from benchmarks import LightningModule, MultiTargetScaler
from benchmarks.cnn import CNN
from dataset.pubchemqc import PubChemQCDataset

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "pubchemqc_homo",
    "pubchemqc_lumo",
    "pubchemqc_hl_gap",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
]
num_tasks = len(target_properties)
model_path_template = "../models/cnn_1.1M-pac_grid-[PubChemQC]-fold_{fold}"

dataset = PubChemQCDataset(
    properties=target_properties, require_quality_flags=False, num_workers=18
)
targets_array = np.array(dataset.targets, dtype=np.float32)

categories = np.array(
    [f"{pac.category}_{pac.count_total_rings}" for pac in dataset.pacs]
)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), categories), 1
):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=categories[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_subset = Subset(dataset, train_idx)
    test_subset = Subset(dataset, test_idx)

    scaler = MultiTargetScaler()
    scaler.fit(targets_array[train_idx])

    dataset.targets = torch.tensor(scaler.transform(targets_array), dtype=torch.float32)

    train_loader = DataLoader(train_subset, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    model = CNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=92
    )
    load_path = os.path.join(model_path_template.format(fold=fold), "best.ckpt")
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for prop in target_properties:
        # Metric names from trainer.test() start with test_
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]

        print(f"======[ {prop} ]======")
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

    dataset.targets = targets_array

    del model, lightning_model, trainer, scaler
    del train_results, test_results, train_loader, test_loader
    del train_subset, test_subset
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results (mean ± std) ]======")

for prop in target_properties:
    print(f"\n======[ {prop} ]======")
    for dataset_name, key in [("Train", "train"), ("Test", "test")]:
        m = all_results[key][prop]
        print(f"{dataset_name}:")
        print(f" + MAE: {np.mean(m['mae']):.4f} ± {np.std(m['mae'], ddof=1):.4f}")
        print(f" + RMSD: {np.mean(m['rmsd']):.4f} ± {np.std(m['rmsd'], ddof=1):.4f}")
        print(f" + Max: {np.mean(m['max']):.4f} ± {np.std(m['max'], ddof=1):.4f}")
        print(f" + Min: {np.mean(m['min']):.4f} ± {np.std(m['min'], ddof=1):.4f}")
        print(f" + R^2: {np.mean(m['r2']):.4f} ± {np.std(m['r2'], ddof=1):.4f}")

Using device: cuda


D:\Coding\pac-gcnn\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lig


======[ Evaluating fold 1/5 ]======
======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


======[ Test Set Evaluation ]======
======[ pubchemqc_homo ]======
 + Train -> MAE: 0.0267	RMSD: 0.0319	Max: 0.1221	Min: 0.0000	R^2: 0.9986
 + Test -> MAE: 0.0680	RMSD: 0.1145	Max: 1.3414	Min: 0.0000	R^2: 0.9811
======[ pubchemqc_lumo ]======
 + Train -> MAE: 0.0180	RMSD: 0.0229	Max: 0.1838	Min: 0.0000	R^2: 0.9991
 + Test -> MAE: 0.0728	RMSD: 0.1355	Max: 1.8308	Min: 0.0001	R^2: 0.9675
======[ pubchemqc_hl_gap ]======
 + Train -> MAE: 0.0245	RMSD: 0.0314	Max: 0.2012	Min: 0.0000	R^2: 0.9985
 + Test -> MAE: 0.1039	RMSD: 0.1950	Max: 1.8418	Min: 0.0000	R^2: 0.9442


...

======[ Test Set Evaluation ]======
======[ pubchemqc_homo ]======
 + Train -> MAE: 0.0084	RMSD: 0.0106	Max: 0.0709	Min: 0.0000	R^2: 0.9998
 + Test -> MAE: 0.0674	RMSD: 0.1177	Max: 1.1272	Min: 0.0000	R^2: 0.9805
======[ pubchemqc_lumo ]======
 + Train -> MAE: 0.0078	RMSD: 0.0100	Max: 0.0658	Min: 0.0000	R^2: 0.9998
 + Test -> MAE: 0.0733	RMSD: 0.1383	Max: 1.6753	Min: 0.0000	R^2: 0.9684
======[ pubchemqc_hl_gap ]======
 + Train -> MAE: 0.0117	RMSD: 0.0145	Max: 0.1125	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.1037	RMSD: 0.2042	Max: 2.5605	Min: 0.0002	R^2: 0.9375


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Evaluating fold 5/5 ]======
======[ Train Set Evaluation ]======


D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


======[ Test Set Evaluation ]======
======[ pubchemqc_homo ]======
 + Train -> MAE: 0.0079	RMSD: 0.0100	Max: 0.0680	Min: 0.0000	R^2: 0.9999
 + Test -> MAE: 0.0683	RMSD: 0.1252	Max: 1.2256	Min: 0.0001	R^2: 0.9784
======[ pubchemqc_lumo ]======
 + Train -> MAE: 0.0065	RMSD: 0.0084	Max: 0.0827	Min: 0.0000	R^2: 0.9999
 + Test -> MAE: 0.0701	RMSD: 0.1335	Max: 1.7331	Min: 0.0000	R^2: 0.9700
======[ pubchemqc_hl_gap ]======
 + Train -> MAE: 0.0070	RMSD: 0.0089	Max: 0.0748	Min: 0.0000	R^2: 0.9999
 + Test -> MAE: 0.1047	RMSD: 0.2078	Max: 2.6466	Min: 0.0001	R^2: 0.9356

======[ Final k-fold results (mean ± std) ]======

======[ pubchemqc_homo ]======
Train:
 + MAE: 0.0153 ± 0.0082
 + RMSD: 0.0186 ± 0.0098
 + Max: 0.0881 ± 0.0258
 + Min: 0.0000 ± 0.0000
 + R^2: 0.9994 ± 0.0006
Test:
 + MAE: 0.0692 ± 0.0027
 + RMSD: 0.1224 ± 0.0079
 + Max: 1.2730 ± 0.2869
 + Min: 0.0000 ± 0.0000
 + R^2: 0.9793 ± 0.0023

======[ pubchemqc_lumo ]======
Train:
 + MAE: 0.0104 ± 0.0045
 + RMSD: 0.0131 ± 0.0057
 + Max: 